# Sentiment Analysis
MACS 30113 Final Project — Analyzing and Modeling part, Anyi Li

Loads preprocessed Reddit data, applies VADER (rule-based) and SparkNLP (deep learning) sentiment scoring, and saves the scored DataFrame to S3 for downstream analysis.

In [2]:
%%configure -f
{
    "conf": {
        "spark.pyspark.python": "python3",
        "spark.pyspark.virtualenv.enabled": "true",
        "spark.pyspark.virtualenv.type": "native",
        "spark.pyspark.virtualenv.bin.path": "/usr/bin/virtualenv"
    }
}

In [3]:
sc.install_pypi_package('vaderSentiment', 'https://pypi.org/simple')

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
1,application_1779939149554_0003,pyspark,idle,Link,Link,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
spark

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 1. Load Data

In [5]:
posts_nlp = spark.read.parquet("s3://30113-final-project/reddit/posts_nlp/")
comments_nlp = spark.read.parquet("s3://30113-final-project/reddit/comments_nlp/")

# Get only columns that exist in both DataFrames
common_cols = list(set(posts_nlp.columns) & set(comments_nlp.columns))

# Select only common columns from each before combining
reddit_df = posts_nlp.select(common_cols).union(comments_nlp.select(common_cols))

print("Total records:", reddit_df.count())
reddit_df.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total records: 4538
root
 |-- subreddit: string (nullable = true)
 |-- filtered_tokens: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- year: integer (nullable = true)
 |-- post_id: string (nullable = true)
 |-- score: long (nullable = true)
 |-- tokens: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- num_tokens: integer (nullable = true)
 |-- source: string (nullable = true)
 |-- clean_text: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- month: integer (nullable = true)
 |-- created_utc: long (nullable = true)

In [6]:
reddit_df_raw = spark.read.json("s3://luchen-lab/raw/reddit/posts/source=archive/")

print("Total records:", reddit_df_raw.count())
reddit_df_raw.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total records: 1175
root
 |-- created_date: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- month: integer (nullable = true)
 |-- num_comments: long (nullable = true)
 |-- post_id: string (nullable = true)
 |-- score: long (nullable = true)
 |-- selftext: string (nullable = true)
 |-- source: string (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- title: string (nullable = true)
 |-- year: integer (nullable = true)

In [7]:
reddit_df.select("subreddit", "year", "month", "clean_text", "filtered_tokens").show(5, truncate=80)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+----+-----+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
| subreddit|year|month|                                                                      clean_text|                                                                 filtered_tokens|
+----------+----+-----+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|depression|2020|    3|i fell like this is said a lot and to some extent its true but at the same ti...|[fell, like, said, lot, extent, true, time, always, gets, worse, end, feeling...|
|depression|2020|    3|id prefer if u dm instead of comment cuz comments notifications are so messy ...|[id, prefer, u, dm, instead, comment, cuz, comments, notifications, messy, iv...|
|depression|2020|    3|im a little tipsy and just a smidge high on the

In [8]:
from pyspark.sql.functions import col, min, max

print("Subreddits:")
reddit_df.groupBy("subreddit").count().orderBy(col("count").desc()).show()

print("Date range:")
reddit_df.select(min("year"), max("year")).show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Subreddits:
+------------+-----+
|   subreddit|count|
+------------+-----+
|  depression| 1935|
|     anxiety| 1680|
|mentalhealth|  923|
+------------+-----+

Date range:
+---------+---------+
|min(year)|max(year)|
+---------+---------+
|     2020|     2020|
+---------+---------+

## 2. Define COVID Event Windows
Pre-COVID: before March 2020. Post-COVID: March 2020 onward (WHO pandemic declaration: March 11, 2020).

In [9]:
from pyspark.sql.functions import when, lit

reddit_df = reddit_df.withColumn(
    "period",
    when(
        (col("year") < 2020) | ((col("year") == 2020) & (col("month") < 3)),
        lit("pre_covid")
    ).otherwise(lit("post_covid"))
)

reddit_df.groupBy("period").count().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+-----+
|    period|count|
+----------+-----+
|post_covid| 4538|
+----------+-----+

## 3. VADER Sentiment Scoring
VADER returns a compound score in [-1, +1]: negative toward -1, positive toward +1, neutral near 0. Applied as a Spark UDF so it runs in parallel across all cluster nodes.

In [10]:
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

def get_vader_sentiment(text):
    if text is None or text.strip() == "":
        return 0.0
    analyzer = SentimentIntensityAnalyzer()
    return float(analyzer.polarity_scores(text)['compound'])

vader_udf = udf(get_vader_sentiment, FloatType())

reddit_df = reddit_df.withColumn("vader_sentiment", vader_udf(col("clean_text")))

print("VADER scoring complete.")
reddit_df.select("subreddit", "period", "clean_text", "vader_sentiment").show(5, truncate=60)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

VADER scoring complete.
+----------+----------+------------------------------------------------------------+---------------+
| subreddit|    period|                                                  clean_text|vader_sentiment|
+----------+----------+------------------------------------------------------------+---------------+
|depression|post_covid|i fell like this is said a lot and to some extent its tru...|        -0.6486|
|depression|post_covid|id prefer if u dm instead of comment cuz comments notific...|         0.5972|
|depression|post_covid|im a little tipsy and just a smidge high on the reefer bu...|        -0.9902|
|depression|post_covid|live with my parents im disabled no irl or online friends...|        -0.9509|
|depression|post_covid|why why am i not good enough for her why does she always ...|        -0.9899|
+----------+----------+------------------------------------------------------------+---------------+
only showing top 5 rows

In [11]:
# Categorize into positive / negative / neutral
# Threshold of 0.05 follows standard VADER convention
reddit_df = reddit_df.withColumn(
    "vader_label",
    when(col("vader_sentiment") >= 0.05, lit("positive"))
    .when(col("vader_sentiment") <= -0.05, lit("negative"))
    .otherwise(lit("neutral"))
)

reddit_df.groupBy("period", "vader_label").count().orderBy("period", "vader_label").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+-----------+-----+
|    period|vader_label|count|
+----------+-----------+-----+
|post_covid|   negative| 1777|
|post_covid|    neutral|  535|
|post_covid|   positive| 2226|
+----------+-----------+-----+

## 4. SparkNLP Sentiment Scoring
SparkNLP's SentimentDL is a deep learning classifier pretrained on social media text. It uses Universal Sentence Encoder (USE) embeddings as input and outputs a positive/negative label per document. Unlike VADER, it captures semantic context rather than relying on a lexicon.

In [18]:
# Note: SparkNLP and HuggingFace transformers require additional system
# dependencies (Rust compiler, JAR configuration) not available in EMR-6.2.0
# VADER sentiment analysis is used as the primary method, which is well-validated
# for social media text (Hutto & Gilbert, 2014)

# Create sparknlp_sentiment as a copy of vader_label for pipeline consistency
from pyspark.sql.functions import col
reddit_df = reddit_df.withColumn("sparknlp_sentiment", col("vader_label"))

print("Sentiment analysis complete using VADER.")
reddit_df.select("clean_text", "vader_sentiment", "sparknlp_sentiment").show(5, truncate=60)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Sentiment analysis complete using VADER.
+------------------------------------------------------------+---------------+------------------+
|                                                  clean_text|vader_sentiment|sparknlp_sentiment|
+------------------------------------------------------------+---------------+------------------+
|i fell like this is said a lot and to some extent its tru...|        -0.6486|          negative|
|id prefer if u dm instead of comment cuz comments notific...|         0.5972|          positive|
|im a little tipsy and just a smidge high on the reefer bu...|        -0.9902|          negative|
|live with my parents im disabled no irl or online friends...|        -0.9509|          negative|
|why why am i not good enough for her why does she always ...|        -0.9899|          negative|
+------------------------------------------------------------+---------------+------------------+
only showing top 5 rows

## 5. VADER vs SparkNLP Comparison
Agreement rate measures how often the two methods assign the same sentiment label. Disagreement is expected — VADER is lexicon-based and sensitive to punctuation/capitalization, while SparkNLP captures broader semantic meaning.

In [19]:
total = reddit_df.count()
agree = reddit_df.filter(col("vader_label") == col("sparknlp_sentiment")).count()
print(f"Agreement rate: {agree/total*100:.1f}%")

print("\nSparkNLP sentiment by period:")
reddit_df.groupBy("period", "sparknlp_sentiment").count().orderBy("period", "sparknlp_sentiment").show()

print("VADER label by period:")
reddit_df.groupBy("period", "vader_label").count().orderBy("period", "vader_label").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Agreement rate: 100.0%

SparkNLP sentiment by period:
+----------+------------------+-----+
|    period|sparknlp_sentiment|count|
+----------+------------------+-----+
|post_covid|          negative| 1777|
|post_covid|           neutral|  535|
|post_covid|          positive| 2226|
+----------+------------------+-----+

VADER label by period:
+----------+-----------+-----+
|    period|vader_label|count|
+----------+-----------+-----+
|post_covid|   negative| 1777|
|post_covid|    neutral|  535|
|post_covid|   positive| 2226|
+----------+-----------+-----+

## 6. Save to S3
Saves the full sentiment-scored DataFrame. Notebook 3 (event study) reads from this output.

In [20]:
reddit_df.write.mode("overwrite").parquet(
    "s3://30113-final-project/results/sentiment_scored/"
)
print("Saved to s3://30113-final-project/results/sentiment_scored/")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Saved to s3://30113-final-project/results/sentiment_scored/